In [ ]:
!git clone https://github.com/Tahoni01/Continual-hate-speech-detection.git
%cd Continual-hate-speech-detection

In [ ]:
%pip install -r requirements.txt

In [ ]:
import torch
from dataset.df_loader import (getdf_davidson, getdf_hatexplain)
from dataset.stream_generator import (create_continual_stream, online_stream)
from transformers import (AutoTokenizer, AutoConfig)
from models.model_builder import CustomClassifier
from models.trainer import ContinualTrainer

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", device)

In [ ]:
print("Loading datasets...")

df_dv = getdf_davidson()
df_hx = getdf_hatexplain()

print("Davidson:", df_dv.shape)
print("HateXplain:", df_hx.shape)

In [ ]:
full_stream = create_continual_stream(
    df_list=[df_dv, df_hx],
    batch_size=10
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilroberta-base")

model = CustomClassifier(
    model_name="distilroberta-base",
    num_labels=3,
    )

model.to(device)

In [ ]:
import matplotlib.pyplot as plt
from models.trainer import ContinualTrainer
from strategy.replay import ReplayStrategy
from utils.metrics import compute_accuracy, compute_f1
from utils.plot_utils import plot_loss, plot_batch_metrics, plot_conf_matrix, plot_classwise_accuracy, plot_prediction_distribution

batch_size = 10

# label map
label_map = {"hatespeech": 0, "offensive": 1, "normal": 2}

# Strategy
strategy = ReplayStrategy(model=model, buffer_size=50)  # buffer piccolo per test

# Trainer
trainer = ContinualTrainer(
    model=model,
    tokenizer=tokenizer,
    device=device,
    batch_size=batch_size,
    strategy=strategy  # <-- plug and play
)

# Test stream
test_stream = full_stream[:50]  # 2 batch da 10 elementi ciascuno, solo per test

# Train
losses, preds, labels = trainer.train_continual(full_stream, log_every=1)

# Metriche globali
flat_preds = torch.cat(preds)
flat_labels = torch.cat(labels)

acc = compute_accuracy(flat_preds, flat_labels)
f1 = compute_f1(flat_preds, flat_labels)

print(f"\nAccuracy globale: {acc:.4f}, F1 globale: {f1:.4f}")

# Predizioni batch
for i, (p, l) in enumerate(zip(preds, labels)):
    print(f"\nBatch {i+1} predizioni:", p.tolist())
    print(f"Batch {i+1} etichette:", l.tolist())

plot_loss(losses)
plot_batch_metrics(preds, labels)
plot_conf_matrix(preds, labels, label_map)
plot_classwise_accuracy(preds, labels, label_map)
#plot_prediction_distribution(preds, label_map)